In [0]:
%run "../../utils/00_config"

In [0]:
%run "../../utils/02_data_quality"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

stg_catalog = spark.table(
    get_table("staging", "stg_wfm_catalog")
)

print(f"Source rows: {stg_catalog.count()}")

In [0]:
window_latest = (
    Window
    .partitionBy("item_id")
    .orderBy(
        F.col("observed_at").desc_nulls_last(),
        F.col("execution_id").desc()
    )
)

dim_item = (
    stg_catalog
    .withColumn("_row_number", F.row_number().over(window_latest))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
dim_item = dim_item.select(
    "item_id",
    "item_slug",
    "item_name",
    "product_slug",
    "category",
    "game_ref",
    "tags",
    "is_set_root",
    "set_part_ids",
    "quantity_in_set",
    "ducats",
    "required_mastery_rank",
    "trading_tax",
    "is_tradable",
    "description",
    "wiki_url",
    "icon_path",
    "thumbnail_path",
    "sub_icon_path",
    F.col("execution_id").alias("source_execution_id"),
    F.col("observed_at").alias("catalog_observed_at")
)

In [0]:
# Data quality checks
check_required_fields(
    dim_item,
    [
        "item_id",
        "item_slug",
        "item_name",
        "product_slug",
        "category",
        "catalog_observed_at"
    ]
)

check_unique_keys(dim_item, ["item_id"])

print(f"Dimension rows: {dim_item.count()}")

# display(
#     dim_item.orderBy("product_slug", "is_set_root", "item_slug")
# )

In [0]:
# Merge into Delta table
'''
- Item inexistente: insere
- Item existente com informação mais recente: atualiza
- Item existente com informação mais antiga: preserva a versão atual
- item_id -> funciona como chave da dimensão
- product_slug -> identifica o conjunto de origem
- is_set_root -> diferencia os itens principais dos componentes
- set_part_ids -> preserva a composição de cada conjunto
- source_execution_id e catalog_observed_at -> rastreamento da origem dos atributos
'''
from delta.tables import DeltaTable

target_table = get_table("intermediate", "dim_item")

if spark.catalog.tableExists(target_table):
    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(
            dim_item.alias("source"),
            "target.item_id = source.item_id"
        )
        .whenMatchedUpdateAll(
            condition="""
                source.catalog_observed_at >= target.catalog_observed_at
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        dim_item.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )

print(f"Table materialized: {target_table}")

In [0]:
# Final validation

dim_item_table = spark.table(target_table)

check_required_fields(
    dim_item_table,
    [
        "item_id",
        "item_slug",
        "item_name",
        "product_slug",
        "category",
        "catalog_observed_at"
    ]
)

check_unique_keys(dim_item_table, ["item_id"])

print(f"Total records: {dim_item_table.count():,}")
print(
    f"Distinct items: "
    f"{dim_item_table.select('item_id').distinct().count():,}"
)

# display(
#     dim_item_table
#     .orderBy("product_slug", "is_set_root", "item_slug")
# )